In [ ]:
%load_ext tensorboard

In [ ]:
import os
import numpy as np
import tensorflow as tf
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [ ]:
data_dir = 'D:\\Projects\\dl-tl-mini-project\\assets\\data'
arabic_alphabet = ['?', '??', '?', '?', '?', '?', '?', '?', '?', '?', '?', '?', '?', '?', '?', '??', '?', '?', '?', '?', '?', '?', '?', '?', '?', '?', '?', '?', '?', '?', '?', '?']
arabic_alphabet_latin = os.listdir(f'{data_dir}\\ArASL_Database_54K')

In [ ]:
def load_image(image_path):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(64, 64))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = tf.keras.applications.efficientnet.preprocess_input(img_array)
    return img_array

In [ ]:
data = pd.read_csv(f'{data_dir}\\ArSL_Data_Labels.csv')
data.drop(columns=['#'], inplace=True)
data.columns = ['image', 'class']
data

In [ ]:
print(data['class'])

In [ ]:
X = []
y = []

for i in range(data.shape[0]):
    img_path = os.path.join(data_dir + '\\ArASL_Database_54K\\' + data['class'][i], data['image'][i])
    img = load_image(img_path)
    X.append(img)
    y.append(arabic_alphabet_latin.index(data['class'][i]))    
    print(f'Processed images: {i+1}/{data.shape[0]}', end='\r')

print("\n")

X = np.array(X)
X = X.reshape((54049, 64, 64, 3))
y = np.array(y)

print(X.shape, y.shape)

In [ ]:
"""base_model = tf.keras.applications.EfficientNetB7(input_shape=(64, 64, 3), include_top=False, weights=None)

model = tf.keras.models.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    # tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(2560, activation='relu'),
    tf.keras.layers.Dense(32, activation='softmax')
])

model.load_weights('D:\\Projects\\dl-tl-mini-project\\model\\model_checkpoint2.keras')
model.summary()"""

model = tf.keras.models.load_model('D:\\Projects\\dl-tl-mini-project\\model\\model.keras')
model.summary()

In [ ]:
# evaluate the model
y_pred = model.predict(X)

# confusion matrix
cm = confusion_matrix(y, y_pred.argmax(axis=1))

# plot the confusion matrix
plt.figure(figsize=(20, 20))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=arabic_alphabet, yticklabels=arabic_alphabet)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# accuracy
from sklearn.metrics import accuracy_score, classification_report
accuracy = accuracy_score(y, y_pred.argmax(axis=1))
print(f'Accuracy: {accuracy}')
print(classification_report(y, y_pred.argmax(axis=1), target_names=arabic_alphabet))